In [ ]:

# imports
import pandas as pd
from tqdm import tqdm
import numpy as np
import json
import os
from openai import OpenAI, AsyncOpenAI, OpenAIError
import asyncio
from tqdm.asyncio import tqdm_asyncio

from miti_global_scores import MAIN_PROMPT, MITI_GUIDE

PROJECT_DIR = os.getenv("PROJECT_DIR")
api_key = os.getenv("OPENAI_API_KEY")

################################################
##### Interview transcripts   ##################
################################################

data = pd.read_csv("/path/to/project/data/raw/main_socialmedia/chats_raw.csv", low_memory=False)
data = data.sort_values(by=["session_id", "order"], ascending=True)

def construct_transcript(session):
    text = ""
    for _, row in session.iterrows():
        k = row["order"]
        if k % 2 == 1:
            text += f"Clinician: {row['content']}\n"
        else:
            text += f"Client: {row['content']}\n"
    return text

session = data.groupby("session_id", as_index=False).apply(lambda x: construct_transcript(x), include_groups=False).rename(columns={None: "transcript"})




In [ ]:
print(len(session))
print(session.loc[1, "transcript"])

In [ ]:
session.head()

In [ ]:
prefix = "Clinician: Hi! In this interview I want to learn more about how you spend your time."
treated_sessions= session[~session["transcript"].str.startswith(prefix, na=False)].reset_index(drop=True)
print(len(treated_sessions))


In [ ]:


################################################
##### MITI Evaluation Criteria  ################
################################################

# Number of global ratings
components = list(MITI_GUIDE.keys())
n_components = len(components)


# # Test case:
# transcript = session_mi.iloc[0]["transcript"]
# component = "Cultivating Change Talk"


# prompt = MAIN_PROMPT.format(
#     transcript=transcript.strip(),
#     component_name=component.strip(),
#     coding_instructions=MITI_GUIDE[component].strip()
#     )


################################################
#####       API Queries         ################
################################################

# Setup OpenAI
# client = OpenAI(api_key=api_key)

id_name = "session_id"
params = {
    "model": "gpt-5-nano-2025-08-07",
    "text": {"verbosity": "medium"}, # low, medium, high
    "reasoning": {"effort": "low"}, # minimal, low, medium, high
    "store": False,
    "max_output_tokens": 5000,
}

# Prepare the input data for parallel coding (N x M requests, N = number of sessions, M = number of MITI codes)
input_data = []
for _, row in tqdm(treated_sessions.iterrows(), total=treated_sessions.shape[0]):
    for component, instructions in MITI_GUIDE.items():
        identifier = row[id_name]
        prompt = MAIN_PROMPT.format(
            transcript=row["transcript"],
            component_name=component,
            coding_instructions=instructions)
        input_data.append([identifier, component, prompt])

input_data = pd.DataFrame(input_data)
input_data.columns = [id_name, "miti_dimension", "prompt"]

In [ ]:
# PARALLEL REQUESTS
client = AsyncOpenAI(api_key=api_key)

async def classify_row(row, sem, params):
    async with sem:
        identifier = row[id_name]
        component = row["miti_dimension"]
        prompt = row["prompt"]
        try:
            reply = await client.responses.create(input=prompt, **params)
            response = reply.model_dump()
            response_dict = json.loads(response["output"][1]["content"][0]["text"])
            return [identifier, component, response, response_dict["score"], response_dict["justification"], prompt, "0"]
        except Exception as e:
            return [identifier, component, "", -999, -999, prompt, str(e)]

async def main(data, params, n_workers):
    """Execute parallel API requests"""
    sem = asyncio.Semaphore(n_workers)
    tasks = [classify_row(row, sem, params) for _, row in data.iterrows()]
    results = []
    errors = []

    for step in tqdm_asyncio.as_completed(tasks, total=len(tasks)):
        result = await step
        if isinstance(result, list):
            results.append(result)
        else:
            errors.append(result)

    return results, errors


# run (the code below is for running in a jupyter notebook, hence the "await")
results, errors = await main(input_data, params, n_workers=8)

In [ ]:
# Store results
df = pd.DataFrame(results)
df.columns = ["session_id", "miti_dimension", "gpt_response", "score", "justification", "prompt", "error"]

In [ ]:
date = pd.Timestamp.now().strftime("%Y%m%d")

In [ ]:
df.to_csv(f"output/w2_miti_global_scores_{date}_v003.csv", index=False)

In [ ]:
df.groupby("miti_dimension")["score"].value_counts().sort_index()